# 04 · Cobertura de fotos de jugadores en Wikidata

Comprueba qué jugadores del modelo tienen foto en Wikidata (propiedad P18). **No descarga ninguna imagen**:
solo consulta los metadatos (url, licencia, autor y tamaño).

**Cómo se acepta una entidad de Wikidata.** Un homónimo con foto es peor que un hueco, así que una entidad
solo se acepta si cumple todo esto:

1. Es una persona (P31 = Q5) con ocupación futbolista (P106 = Q937857).
2. Consta como jugador (P54) de uno de los clubes del jugador en esta temporada, en una etapa sin fecha de fin
   o que termina en 2025 o después.
3. Su nombre (etiqueta o alias en español, inglés, catalán, euskera o gallego) coincide, sin acentos ni
   mayúsculas, con el nombre del jugador en alguna de las tres fuentes del proyecto.
4. Es la única entidad que cumple 1, 2 y 3. Si hay dos, no se acepta ninguna.
5. Si Wikidata tiene fecha de nacimiento, cuadra con la edad que da WhoScored (±1 año).

Las consultas se guardan en `data/cache_wikidata.json` y no se repiten al volver a ejecutar el cuaderno.

In [1]:
# Instalamos las librerías del cuaderno; si ya están instaladas no cambia nada
%pip install -q pandas requests unidecode

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importamos os para situarnos en la raíz del proyecto
import os
# Importamos json para la caché
import json
# Importamos time para espaciar las consultas
import time
# Importamos re para limpiar el HTML del autor
import re
# Importamos requests para consultar Wikidata y Wikimedia Commons
import requests
# Importamos pandas para trabajar con tablas
import pandas as pd
# Importamos unidecode para comparar nombres sin acentos
from unidecode import unidecode

# Si el cuaderno se ejecuta desde notebooks/, subimos a la raíz para usar rutas relativas
if os.path.basename(os.getcwd()) == "notebooks":
    # Cambiamos a la carpeta superior
    os.chdir("..")

# Ensanchamos la salida para que las tablas no se partan
pd.set_option("display.width", 250)

## 1. Clubes en Wikidata

Identificadores comprobados a mano: club de fútbol (P31) con país España (P17) y nombre del primer equipo
(no el filial ni el femenino).

In [3]:
# Equipo del modelo y su entidad de Wikidata
CLUBES = {
    "Alaves": "Q223620", "Athletic Club": "Q8687", "Atletico Madrid": "Q8701", "Barcelona": "Q7156",
    "Celta Vigo": "Q8749", "Elche": "Q10512", "Espanyol": "Q8780", "Getafe": "Q8806", "Girona": "Q11945",
    "Levante": "Q8823", "Mallorca": "Q8835", "Osasuna": "Q10286", "Rayo Vallecano": "Q10300",
    "Real Betis": "Q8723", "Real Madrid": "Q8682", "Real Oviedo": "Q271574", "Real Sociedad": "Q10315",
    "Sevilla": "Q10329", "Valencia": "Q10333", "Villarreal": "Q12297",
}

# Identificación del cliente, como piden las normas de uso de Wikimedia
CABECERAS = {"User-Agent": "scouting-laliga-2526/1.0 (proyecto de datos; python-requests)"}
# Fichero de caché
RUTA_CACHE = "data/cache_wikidata.json"
# Cargamos la caché si existe
cache = json.load(open(RUTA_CACHE, encoding="utf-8")) if os.path.exists(RUTA_CACHE) else {}


# Hace una consulta GET con caché y reintentos si el servidor pide esperar
def consultar(url, params):
    # Clave de la caché: url y parámetros ordenados
    clave = url + "?" + json.dumps(params, sort_keys=True, ensure_ascii=False)
    # Si ya la tenemos, no se repite
    if clave in cache:
        return cache[clave]
    # Hasta 8 intentos
    for _ in range(8):
        # Petición
        r = requests.get(url, params=params, headers=CABECERAS, timeout=120)
        # Respuesta correcta
        if r.status_code == 200:
            # Guardamos en la caché
            cache[clave] = r.json()
            # Escribimos la caché en disco
            json.dump(cache, open(RUTA_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
            # Pausa de cortesía
            time.sleep(1)
            # Devolvemos la respuesta
            return cache[clave]
        # Si el servidor pide esperar, esperamos lo que diga
        time.sleep(int(r.headers.get("retry-after", 10)))
    # Sin respuesta tras los reintentos
    raise RuntimeError(f"sin respuesta de {url}")


# Consulta SPARQL de los futbolistas de un club con etapa vigente en 2025 o después
SPARQL = """
SELECT ?p ?nombre ?nacimiento ?imagen WHERE {
  ?p p:P54 ?etapa . ?etapa ps:P54 wd:%s .
  OPTIONAL { ?etapa pq:P582 ?fin }
  FILTER(!BOUND(?fin) || YEAR(?fin) >= 2025)
  ?p wdt:P31 wd:Q5 ; wdt:P106 wd:Q937857 .
  OPTIONAL { ?p wdt:P569 ?nacimiento }
  OPTIONAL { ?p wdt:P18 ?imagen }
  { ?p rdfs:label ?nombre } UNION { ?p skos:altLabel ?nombre }
  FILTER(LANG(?nombre) IN ("es", "en", "ca", "eu", "gl"))
}"""

# Candidatos de todos los clubes
filas = []
# Recorremos los clubes
for equipo, qid in CLUBES.items():
    # Consulta al servicio SPARQL de Wikidata
    res = consultar("https://query.wikidata.org/sparql", {"query": SPARQL % qid, "format": "json"})
    # Recorremos los resultados
    for b in res["results"]["bindings"]:
        # Guardamos una fila por entidad y nombre
        filas.append({"equipo": equipo, "id_wikidata": b["p"]["value"].rsplit("/", 1)[1],
                      "nombre_wd": b["nombre"]["value"], "nacimiento": b.get("nacimiento", {}).get("value"),
                      "imagen": b.get("imagen", {}).get("value")})
# Pasamos a tabla
candidatos = pd.DataFrame(filas).drop_duplicates()
# Mostramos el volumen
print("entidades candidatas:", candidatos["id_wikidata"].nunique(), "· filas nombre-club:", len(candidatos))

entidades candidatas: 3300 · filas nombre-club: 10573


## 2. Emparejamiento con los jugadores del modelo

In [4]:
# Función que deja un nombre en minúsculas, sin acentos, puntos ni guiones (la del cuaderno 00)
def normalizar(nombre):
    # Quitamos acentos, pasamos a minúsculas y eliminamos puntos y guiones
    n = unidecode(str(nombre)).lower().replace(".", "").replace("-", " ")
    # Compactamos los espacios repetidos
    return " ".join(n.split())


# Nombre normalizado de cada candidato
candidatos["n"] = candidatos["nombre_wd"].map(normalizar)

# Jugadores del modelo
jug = pd.read_csv("data/modelo/dim_jugador.csv", dtype={"id_whoscored": "Int64"})
# Equipos del modelo
equipos = pd.read_csv("data/modelo/dim_equipo.csv")
# Posiciones y grupos
posiciones = pd.read_csv("data/modelo/dim_posicion.csv")
# Hechos, para saber quién tiene percentil fiable
hechos = pd.read_csv("data/modelo/hechos_jugador.csv")
# Mapa de nombres: nombre en cada fuente
mapa = pd.read_csv("data/mapa_nombres.csv", dtype={"id_whoscored": "Int64"})
# Alineaciones de WhoScored: edad del jugador
alineaciones = pd.read_csv("data/origen/alineaciones_laliga_2526.csv")

# Añadimos equipo, grupo y si su percentil es fiable
jug = (jug.merge(equipos[["id_equipo", "equipo"]], on="id_equipo")
          .merge(posiciones[["id_posicion", "grupo"]], on="id_posicion")
          .merge(hechos[["id_jugador", "percentil_fiable"]], on="id_jugador")
          .merge(mapa[["id_understat", "nombre_understat", "nombre_whoscored", "equipo_understat"]], on="id_understat"))
# Edad según WhoScored (la mayor de la temporada)
edad = alineaciones.groupby("player_id")["edad"].max()
# La añadimos por identificador de WhoScored
jug["edad_ws"] = jug["id_whoscored"].map(edad)


# Clubes de un jugador en la temporada: el del modelo, el de Understat y los de sus eventos
def clubes(fila):
    # Empezamos por el del modelo y el de Understat
    c = {fila["equipo"], fila["equipo_understat"]}
    # Añadimos los clubes de "Celta Vigo 10 + Girona 17"
    if pd.notna(fila["equipos_temporada"]):
        # Quitamos el número de partidos de cada parte
        c |= {re.sub(r"\s+\d+$", "", parte) for parte in fila["equipos_temporada"].split(" + ")}
    # Devolvemos el conjunto
    return c


# Decide la entidad de Wikidata de un jugador; devuelve (id, motivo)
def emparejar(fila):
    # Nombres del jugador en las tres fuentes
    nombres = {normalizar(x) for x in [fila["jugador"], fila["nombre_understat"], fila["nombre_whoscored"]] if pd.notna(x)}
    # Candidatos de sus clubes con alguno de esos nombres
    c = candidatos[candidatos["equipo"].isin(clubes(fila)) & candidatos["n"].isin(nombres)]
    # Entidades distintas
    ids = c["id_wikidata"].unique()
    # Ninguna
    if len(ids) == 0:
        return None, "sin coincidencia en sus clubes"
    # Más de una: no se elige
    if len(ids) > 1:
        return None, "ambiguo: " + ", ".join(sorted(ids))
    # Fecha de nacimiento de la entidad
    nac = c["nacimiento"].dropna()
    # Si hay fecha y edad, tienen que cuadrar
    if len(nac) and pd.notna(fila["edad_ws"]):
        # Año de nacimiento
        anio = int(nac.iloc[0][:4])
        # Edad que tendría en 2025 frente a la de WhoScored
        if abs((2025 - anio) - fila["edad_ws"]) > 1:
            return None, f"edad no cuadra: nacido en {anio}, WhoScored dice {int(fila['edad_ws'])} años"
    # Aceptada
    return ids[0], "aceptado"


# Emparejamos a los 561
jug[["id_wikidata", "motivo"]] = jug.apply(emparejar, axis=1, result_type="expand")
# Recuento de motivos
print(jug["motivo"].str.split(":").str[0].value_counts().to_string())

motivo
aceptado                          293
sin coincidencia en sus clubes    267
edad no cuadra                      1


In [5]:
# Casos ambiguos y de edad que no cuadra, para revisarlos
print(jug[jug["motivo"].str.startswith(("ambiguo", "edad"))][["jugador", "equipo", "motivo"]].to_string(index=False))

     jugador     equipo                                                 motivo
Hugo Álvarez Celta Vigo edad no cuadra: nacido en 1985, WhoScored dice 23 años


## 3. Metadatos de las fotos en Wikimedia Commons

In [6]:
# Foto de cada entidad aceptada (si tiene varias, la primera)
fotos = candidatos.dropna(subset=["imagen"]).groupby("id_wikidata")["imagen"].first()
# Url de la foto
jug["imagen"] = jug["id_wikidata"].map(fotos)
# Nombre del fichero en Commons
jug["fichero"] = jug["imagen"].map(lambda u: "File:" + requests.utils.unquote(u.rsplit("/", 1)[1]) if isinstance(u, str) else None)

# Ficheros que hay que consultar
ficheros = sorted(jug["fichero"].dropna().unique())
# Metadatos por fichero
meta = {}
# Consultamos de 50 en 50, el máximo de la API
for i in range(0, len(ficheros), 50):
    # Consulta de información de imagen
    res = consultar("https://commons.wikimedia.org/w/api.php", {
        "action": "query", "titles": "|".join(ficheros[i:i + 50]), "prop": "imageinfo",
        "iiprop": "url|size|extmetadata", "format": "json", "formatversion": "2"})
    # Normalizaciones de título que hace la API
    norm = {n["from"]: n["to"] for n in res["query"].get("normalized", [])}
    # Recorremos las páginas
    for p in res["query"]["pages"]:
        # Información de la imagen
        ii = (p.get("imageinfo") or [{}])[0]
        # Metadatos extendidos
        em = ii.get("extmetadata", {})
        # Guardamos lo que interesa
        meta[p["title"]] = {"url_imagen": ii.get("url"), "ancho": ii.get("width"), "alto": ii.get("height"),
                            "licencia": em.get("LicenseShortName", {}).get("value"),
                            "autor": re.sub(r"<[^>]+>", "", em.get("Artist", {}).get("value", "")).strip() or None}
    # Asociamos también los títulos sin normalizar
    for a, b in norm.items():
        # Mismo metadato con el título original
        meta[a] = meta.get(b, {})

# Añadimos los metadatos a cada jugador
for col in ["url_imagen", "licencia", "autor", "ancho", "alto"]:
    # Valor del metadato o vacío
    jug[col] = jug["fichero"].map(lambda f: meta.get(f, {}).get(col) if isinstance(f, str) else None)

## 4. Guardado e informe

In [7]:
# Columnas del fichero
COLUMNAS = ["id_jugador", "jugador", "equipo", "id_wikidata", "url_imagen", "licencia", "autor", "ancho", "alto", "motivo"]
# Guardamos los 561, con o sin foto, para que se vea por qué falta cada una
jug.sort_values("id_jugador")[COLUMNAS].to_csv("data/fotos_wikidata.csv", index=False)

# Tiene foto quien tiene entidad aceptada y url de imagen
jug["tiene_foto"] = jug["url_imagen"].notna()
# Entidad aceptada
jug["aceptado"] = jug["id_wikidata"].notna()
# Resumen general
print(f"entidad de Wikidata aceptada: {int(jug['aceptado'].sum())} de {len(jug)} ({jug['aceptado'].mean() * 100:.1f} %)")
print(f"con foto: {int(jug['tiene_foto'].sum())} de {len(jug)} ({jug['tiene_foto'].mean() * 100:.1f} %)")
# Los que importan en el panel
fiables = jug[jug["percentil_fiable"]]
print(f"con foto entre los de percentil fiable: {int(fiables['tiene_foto'].sum())} de {len(fiables)} ({fiables['tiene_foto'].mean() * 100:.1f} %)")

# Cobertura por grupo de posición
print("\nCOBERTURA POR GRUPO")
print(jug.groupby("grupo").agg(jugadores=("tiene_foto", "size"), con_foto=("tiene_foto", "sum"),
                               pct=("tiene_foto", lambda s: round(s.mean() * 100, 1))).sort_values("pct", ascending=False).to_string())

# Reparto de licencias
print("\nLICENCIAS")
print(jug.loc[jug["tiene_foto"], "licencia"].value_counts(dropna=False).to_string())

# Orientación de las fotos
con_foto = jug[jug["tiene_foto"]]
print(f"\nmás anchas que altas: {int((con_foto['ancho'] > con_foto['alto']).sum())} de {len(con_foto)}")
print(f"cuadradas: {int((con_foto['ancho'] == con_foto['alto']).sum())} · más altas que anchas: {int((con_foto['ancho'] < con_foto['alto']).sum())}")

entidad de Wikidata aceptada: 293 de 561 (52.2 %)
con foto: 218 de 561 (38.9 %)
con foto entre los de percentil fiable: 174 de 368 (47.3 %)

COBERTURA POR GRUPO
                    jugadores  con_foto   pct
grupo                                        
mediapunta                 39        23  59.0
pivote_mediocentro        108        50  46.3
lateral                    94        39  41.5
central                    95        37  38.9
delantero                  83        32  38.6
extremo                    86        31  36.0
sin_confirmar              28         4  14.3
sin_definir                28         2   7.1

LICENCIAS
licencia
CC BY-SA 4.0       102
CC BY-SA 3.0        27
CC BY-SA 2.0        24
CC0                 20
CC BY 2.0           19
CC BY 3.0           15
CC BY 4.0            4
CC BY-SA 3.0 at      3
Public domain        2
Attribution          1
CC BY 3.0 br         1

más anchas que altas: 12 de 218
cuadradas: 1 · más altas que anchas: 205
